# sld-poc — Stage 2: latent diffusion (JAX / TPU)

Robust, self-contained notebook. Every step calls **`load()`**, which force-pulls the latest code from GitHub and re-imports cleanly — so you never fight stale modules again. Everything runs **in-kernel** (a TPU is single-process; no `!python`).

**Before running:** Runtime → Change runtime type → **TPU**. Then run top to bottom.

**Requires Stage A finished** — `codec.pkl` + `latent_stats.npz` in your Drive at `sld-poc-data/`. (The setup cell prints whether `codec.pkl` is found.)

## 1. One-time setup
Mounts Drive, clones the repo, installs deps, and defines `load()`. Run once per session (re-run after a kernel restart).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_DIR = '/content/sld-poc'
DATA_DIR = '/content/drive/MyDrive/sld-poc-data'   # Stage A artifacts live here
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/mccooking/sld-poc.git $REPO_DIR
!pip -q install flax optax datasets tiktoken
os.makedirs(DATA_DIR, exist_ok=True)

def load():
    """Force-pull latest code from GitHub and re-import diffusion cleanly.
    Clears BOTH codec and diffusion from sys.modules so no stale module survives."""
    import sys, subprocess
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', '-q'], check=False)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main', '-q'], check=False)
    src = REPO_DIR + '/src'
    if src not in sys.path: sys.path.insert(0, src)
    for m in ('diffusion', 'codec'): sys.modules.pop(m, None)
    import diffusion
    return diffusion

print('setup done | codec.pkl present:', os.path.exists(DATA_DIR + '/codec.pkl'))

## 2. Check the TPU
Expect a `TpuDevice` and backend `tpu`.

In [ ]:
import jax
print(jax.devices())
print('backend:', jax.default_backend())

## 3. Build latent grids
Encodes TinyStories chunks with the frozen Stage A codec. Forces a clean build (fast + rebuildable). Prints a real shape like `(NNNNNN, 16, 64)`.

In [ ]:
D = load()
P = D.paths(DATA_DIR)
import os
if os.path.exists(P['latents']): os.remove(P['latents'])   # clean rebuild
D.build_latents(P)

## 4. Train the denoiser  ⟵  re-run anytime to resume
Auto-pulls latest code, then trains (resumes from the Drive checkpoint). Watch `mse` trend down — it's noisy, that's normal. First step pauses to JIT-compile.

In [ ]:
D = load()
D.train_diff(D.paths(DATA_DIR))

## 5a. Generate + speed

In [ ]:
D = load()
D.gen(D.paths(DATA_DIR))

## 5b. Infill — fix the ends, generate the middle  *(AR can't do this)*

In [ ]:
D = load()
D.infill(D.paths(DATA_DIR))

## 5c. Self-correct — corrupt one latent, repair from context  *(AR can't do this)*

In [ ]:
D = load()
D.selfcorrect(D.paths(DATA_DIR))